In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

windowed_df = pd.read_csv('../output/windown_sliced_data/windowed_df_10d.csv', parse_dates=['date', 'window_start', 'window_end', 'test_date'])
windowed_df

,date,patient_id,entropy_rate,early_warning_score,sleep_quality_score,agitation_counts,uti_happen,window_start,window_end,test_date,is_test_day
0,2019-04-28,16f4b,0.651879,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
1,2019-04-29,16f4b,0.704693,0.0,1.500000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
2,2019-04-30,16f4b,0.603180,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
3,2019-05-01,16f4b,0.651631,0.0,1.200000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
4,2019-05-02,16f4b,0.668107,0.0,1.000000,0.0,0.0,2019-04-28,2019-05-07,2019-05-08,False
...,...,...,...,...,...,...,...,...,...,...,...
5407,2019-06-21,ec812,0.661363,0.0,1.125000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5408,2019-06-22,ec812,0.623560,0.0,0.666667,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5409,2019-06-23,ec812,0.708682,0.0,0.875000,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False
5410,2019-06-24,ec812,0.656330,0.0,0.888889,0.0,0.0,2019-06-15,2019-06-24,2019-06-25,False


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler

PREDICTORS = ['entropy_rate', 'early_warning_score',
              'sleep_quality_score', 'agitation_counts', 'uti_happen']
n_past = 10
n_features = len(PREDICTORS)

def build_model(n_past, n_features):
    encoder_inputs = tf.keras.layers.Input(shape=(n_past, n_features))
    encoder_outputs, state_h, state_c = tf.keras.layers.LSTM(
        32, return_state=True)(encoder_inputs)  # reduced from 64

    decoder_inputs = tf.keras.layers.RepeatVector(n_past)(encoder_outputs)
    decoder_outputs = tf.keras.layers.LSTM(
        32, return_sequences=True)(decoder_inputs)  # reduced from 64
    decoder_outputs = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(n_features))(decoder_outputs)

    model = tf.keras.Model(encoder_inputs, decoder_outputs)
    model.compile(optimizer='adam', loss='mse')
    return model

In [3]:
results = []

for pid, pid_group in windowed_df.groupby('patient_id'):
    pid_results = []

    # collect all baseline data for this participant
    all_baselines = []
    for (_, window_start), group in pid_group.groupby(['patient_id', 'window_start']):
        baseline = group[group['is_test_day'] == False][PREDICTORS].values
        all_baselines.append(baseline)

    # stack all baselines: shape (n_windows, 10, 5)
    all_baselines = np.array(all_baselines)

    # ADDED: fit scaler on this participant's baseline data only
    scaler = MinMaxScaler()
    all_baselines_2d = all_baselines.reshape(-1, n_features)
    scaler.fit(all_baselines_2d)
    all_baselines_scaled = scaler.transform(all_baselines_2d).reshape(all_baselines.shape)

    # build and fit ONE model per participant on all baselines
    model = build_model(n_past, n_features)
    model.fit(all_baselines_scaled, all_baselines_scaled, epochs=20, batch_size=4, verbose=0)  # ADDED: use scaled

    # score each window
    for (_, window_start), group in pid_group.groupby(['patient_id', 'window_start']):
        baseline = group[group['is_test_day'] == False][PREDICTORS].values
        test_day = group[group['is_test_day'] == True][PREDICTORS].values
        test_date = group[group['is_test_day'] == True]['test_date'].values[0]

        # ADDED: apply the same participant-level scaler
        baseline_scaled = scaler.transform(baseline)
        test_day_scaled = scaler.transform(test_day)

        baseline_input = baseline_scaled.reshape(1, n_past, n_features)
        baseline_reconstructed = model.predict(baseline_input, verbose=0)
        baseline_error = np.mean(np.abs(baseline_input - baseline_reconstructed))

        test_input = np.concatenate(
            [baseline_scaled[1:], test_day_scaled], axis=0
        ).reshape(1, n_past, n_features)
        test_reconstructed = model.predict(test_input, verbose=0)
        test_error = np.mean(np.abs(test_input - test_reconstructed))

        anomaly_score = test_error - baseline_error

        pid_results.append({
            'patient_id': pid,
            'window_start': window_start,
            'test_date': test_date,
            'baseline_error': float(baseline_error),
            'test_error': float(test_error),
            'anomaly_score': float(anomaly_score)
        })

    pid_df = pd.DataFrame(pid_results)
    threshold = pid_df['anomaly_score'].quantile(1 - 0.15)
    pid_df['threshold'] = threshold
    pid_df['anomaly_label'] = pid_df['anomaly_score'].apply(
        lambda x: -1 if x > threshold else 1
    )
    results.append(pid_df)

lstm_results = pd.concat(results).reset_index(drop=True)
print(lstm_results['anomaly_label'].value_counts())
lstm_results

anomaly_label
 1    414
-1     78
Name: count, dtype: int64


,patient_id,window_start,test_date,baseline_error,test_error,anomaly_score,threshold,anomaly_label
0,16f4b,2019-04-28,2019-05-08,0.145082,0.142482,-0.002600,-0.003909,-1
1,16f4b,2019-05-03,2019-05-13,0.171185,0.159862,-0.011322,-0.003909,1
2,1fbe4,2019-04-25,2019-05-05,0.197504,0.189899,-0.007605,0.020904,1
3,1fbe4,2019-04-26,2019-05-06,0.189899,0.185698,-0.004201,0.020904,1
4,1fbe4,2019-04-27,2019-05-07,0.185698,0.200494,0.014796,0.020904,1
...,...,...,...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.168403,0.193873,0.025471,0.009404,-1
488,ec812,2019-06-12,2019-06-22,0.046762,0.057492,0.010729,0.009404,-1
489,ec812,2019-06-13,2019-06-23,0.057490,0.063119,0.005629,0.009404,1
490,ec812,2019-06-14,2019-06-24,0.061634,0.064390,0.002756,0.009404,1


In [4]:
lstm_results.to_csv("../output/Anomaly_delirium_Revised/LSTM_anomaly_labels.csv", index=False)
lstm_results

,patient_id,window_start,test_date,baseline_error,test_error,anomaly_score,threshold,anomaly_label
0,16f4b,2019-04-28,2019-05-08,0.145082,0.142482,-0.002600,-0.003909,-1
1,16f4b,2019-05-03,2019-05-13,0.171185,0.159862,-0.011322,-0.003909,1
2,1fbe4,2019-04-25,2019-05-05,0.197504,0.189899,-0.007605,0.020904,1
3,1fbe4,2019-04-26,2019-05-06,0.189899,0.185698,-0.004201,0.020904,1
4,1fbe4,2019-04-27,2019-05-07,0.185698,0.200494,0.014796,0.020904,1
...,...,...,...,...,...,...,...,...
487,ec812,2019-05-29,2019-06-08,0.168403,0.193873,0.025471,0.009404,-1
488,ec812,2019-06-12,2019-06-22,0.046762,0.057492,0.010729,0.009404,-1
489,ec812,2019-06-13,2019-06-23,0.057490,0.063119,0.005629,0.009404,1
490,ec812,2019-06-14,2019-06-24,0.061634,0.064390,0.002756,0.009404,1


In [6]:
# overall count
n_anomalies = (lstm_results['anomaly_label'] == -1).sum()
n_normal = (lstm_results['anomaly_label'] == 1).sum()
print(f"Anomalies: {n_anomalies}")
print(f"Normal: {n_normal}")
print(f"Anomaly rate: {n_anomalies / len(lstm_results):.2%}")

Anomalies: 78
Normal: 414
Anomaly rate: 15.85%


In [7]:
per_patient = lstm_results.groupby('patient_id').agg(
    n_windows=('anomaly_label', 'count'),
    n_anomalies=('anomaly_label', lambda x: (x == -1).sum())
).reset_index()

per_patient.to_csv("../output/Anomaly_delirium_Revised/LSTM_anomaly_rates_per_person.csv", index=False)
per_patient

,patient_id,n_windows,n_anomalies
0,16f4b,2,1
1,1fbe4,39,6
2,30a32,52,8
3,55cd4,50,8
4,93c14,27,4
5,96adf,39,6
6,a2849,47,7
7,c55f8,71,11
8,c5785,56,9
9,c8574,35,6
